# Day 3 — Knowledge distillation

The teacher (Evo2 embeddings + MLP) is accurate but depends on a 7B-parameter model having
already processed every input. Can we compress what it learned into something tiny that
only looks at k-mer counts?

**Distillation**: train a small "student" not just on the hard 0/1 labels, but on the
teacher's *soft* predictions (its probability, softened by a temperature) — the teacher's
uncertainty carries information ("dark knowledge") that hard labels don't.

In [ ]:
import sys
sys.path.append("../src")

import torch
from data import load_all
from embeddings import load_supervised_embeddings
from featurize import kmer_matrix
from models.classifier_heads import MLPHead
from models.distillation import train_student, distillation_loss
from eval import evaluate_logits, count_params, measure_latency_torch

EMB_DIR = "../../data/supervised/embeddings"
PROCESSED_DIR = "../../data/supervised/processed"

# reload the teacher's training data + a fresh teacher (or re-use the one from notebook 02)
X_train_emb, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")
X_val_emb, y_val, ids_val = load_supervised_embeddings(EMB_DIR, "val")

teacher = MLPHead(d_in=X_train_emb.shape[1])
# NOTE: if you saved teacher weights in notebook 02, load them here instead of retraining.

## Match student inputs to the exact windows the teacher scored

The embeddings were extracted for a subsample of windows (see
`extract_evo2_embeddings.py --max_per_split`), identified by `ids`. We need the *same*
windows' raw sequences to compute k-mer features for the student.

In [ ]:
splits = load_all(PROCESSED_DIR)
train_df = splits["train"].set_index("id")
val_df = splits["val"].set_index("id")

train_seqs = train_df.loc[ids_train, "sequence"].tolist()
val_seqs = val_df.loc[ids_val, "sequence"].tolist()

X_train_kmer = torch.tensor(kmer_matrix(train_seqs, k=4), dtype=torch.float32)
X_val_kmer = torch.tensor(kmer_matrix(val_seqs, k=4), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_train_emb_t = torch.tensor(X_train_emb, dtype=torch.float32)

In [ ]:
# (re-)train the teacher quickly on this exact subsample, then freeze it
optimizer = torch.optim.Adam(teacher.parameters(), lr=1e-3)
n = X_train_emb_t.shape[0]
for epoch in range(20):
    perm = torch.randperm(n)
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            teacher(X_train_emb_t[idx]), y_train_t[idx])
        loss.backward()
        optimizer.step()

teacher.eval()
with torch.no_grad():
    teacher_logits_train = teacher(X_train_emb_t)

## Train the student with the hybrid loss (cross-entropy + soft-target KD)

In [ ]:
student = MLPHead(d_in=X_train_kmer.shape[1], d_hidden=32)  # deliberately tiny

student, history = train_student(
    student, teacher_logits_train, X_train_kmer, y_train_t,
    epochs=30, temperature=4.0, alpha=0.5,
)

In [ ]:
student.eval()
with torch.no_grad():
    val_logits = student(X_val_kmer).numpy()
student_metrics = evaluate_logits(val_logits, y_val)
print("distilled student:", student_metrics)
print("params:", count_params(student), "| latency (ms/sample):",
      measure_latency_torch(student, X_val_kmer[:1]))

### Checkpoint

The student should recover most of the teacher's accuracy while being far smaller (no
Evo2 dependency at inference time at all — just k-mer counts + a tiny MLP) and much
faster. Try varying `temperature` and `alpha` in `train_student` to see the tradeoff.

Next: `04_compression_analysis_and_wrapup.ipynb` — put every model on one chart.